# Experiment 3.16: Eigenvalue Lifting -- one-step $\lambda_{\min}^+$ from matched SGD checkpoints

## Counterpart identity and scope

This notebook is the presentation layer for the paired script:

- `experiments/Tier_1_Core_Mechanism_Tests/EIGENVALUE_LIFTING_per_step/run_experiment.py`

The script is the source of truth for the experiment core. The notebook **imports and runs that script's API** rather than re-implementing the numerical logic.

## What this experiment actually measures

From an SGD warmup trajectory in a tiny deep linear network, the experiment snapshots matched checkpoints and then compares:

1. the Hessian **before** the step,
2. the Hessian after **one SGD step**, and
3. the Hessian after **one Muon step from the same checkpoint**.

The headline floor metric is:

- $\lambda_{\min}^+$ = the **smallest positive** Hessian eigenvalue.

That distinction matters because the Hessian is generally **indefinite** here. This notebook therefore does **not** interpret results as statements about the algebraic minimum eigenvalue of the Hessian.

## Truthful scope framing

This first-pass notebook supports a narrow claim:

- whether a **single Muon step** often lifts $\lambda_{\min}^+$ more than a **single SGD step** from the same SGD checkpoint.

It does **not** by itself establish a full causal decomposition of cumulative Muon-vs-SGD conditioning behavior over full training trajectories.


In [ ]:
from pathlib import Path
import sys
import time
import importlib.util

import numpy as np
import pandas as pd

if "ipykernel" not in sys.modules:
    import matplotlib
    matplotlib.use("Agg")
import matplotlib
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")

EXPERIMENT_REL = Path("experiments/Tier_1_Core_Mechanism_Tests/EIGENVALUE_LIFTING_per_step")
SCRIPT_NAME = "run_experiment.py"


def find_repo_root():
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / EXPERIMENT_REL / SCRIPT_NAME).exists():
            return candidate
    raise FileNotFoundError(
        f"Could not locate {EXPERIMENT_REL / SCRIPT_NAME} by searching upward from {cwd}"
    )


REPO_ROOT = find_repo_root()
EXPERIMENT_DIR = REPO_ROOT / EXPERIMENT_REL
SCRIPT_PATH = EXPERIMENT_DIR / SCRIPT_NAME

spec = importlib.util.spec_from_file_location("eig_lifting_per_step", SCRIPT_PATH)
experiment = importlib.util.module_from_spec(spec)
spec.loader.exec_module(experiment)

print("Repository root:", REPO_ROOT)
print("Experiment directory:", EXPERIMENT_DIR)
print("Counterpart script:", SCRIPT_PATH)
print("NumPy version:", np.__version__)
print("pandas version:", pd.__version__)
print("matplotlib version:", matplotlib.__version__)


## Reproducibility, runtime, and configuration

This cell executes the **shared script API** and records the exact default configuration, seed list, and observed runtime in the current environment.

Notes:

- The script computes a **full central finite-difference Hessian** on the gradient.
- The experiment uses the script's default five-seed configuration.
- Because the notebook calls the same `run_experiment()` function as the script entrypoint, numerical drift between notebook and script should now be minimized.


In [ ]:
t0 = time.perf_counter()
results = experiment.run_experiment(progress=False)
notebook_runtime = time.perf_counter() - t0

rows_df = pd.DataFrame(results["rows"]).sort_values(["step", "seed"]).reset_index(drop=True)
summary_df = pd.DataFrame(results["per_step_summary"]).sort_values("step").reset_index(drop=True)
config_df = pd.DataFrame(sorted(results["config"].items()), columns=["key", "value"])

expected_rows = len(results["seed_list"]) * len(results["config"]["measurement_steps"])
assert len(rows_df) == expected_rows, (len(rows_df), expected_rows)
assert SCRIPT_PATH.resolve() == Path(results["identity"]["script_path"]).resolve()

print(results["identity"]["title"])
print("Question:", results["identity"]["question"])
print("Scope:", results["identity"]["scope_note"])
print()
print("Seeds:", results["seed_list"])
print("Measurement steps:", results["config"]["measurement_steps"])
print("Script-reported runtime: %.3f s" % results["runtime_seconds"])
print("Notebook call runtime:   %.3f s" % notebook_runtime)
print("Rows collected:", len(rows_df))
print()
print("Config:")
with pd.option_context("display.max_colwidth", 120):
    print(config_df.to_string(index=False))
config_df


## Per-step summary table

The table below is the main descriptive summary. Each row aggregates the paired one-step comparisons across seeds at a fixed warmup checkpoint.

Key columns:

- `lambda_min_pos_before_mean`: mean $\lambda_{\min}^+$ before the one-step intervention.
- `lift_sgd_mean`, `lift_muon_mean`: mean one-step multiplicative lifts in $\lambda_{\min}^+$.
- `muon_to_sgd_ratio_of_means`: ratio of the two mean lifts.
- `kappa_ratio_*`: one-step ratio of condition numbers, where values below 1 indicate improved conditioning.
- `n_negative_before_mean`: mean number of negative Hessian eigenvalues before the intervention.


In [ ]:
summary_display = summary_df[[
    "step",
    "n_rows",
    "loss_mean",
    "lambda_min_pos_before_mean",
    "lambda_min_pos_lift_sgd_mean",
    "lambda_min_pos_lift_muon_mean",
    "muon_to_sgd_ratio_of_means",
    "lambda_min_pos_lift_sgd_median",
    "lambda_min_pos_lift_muon_median",
    "kappa_ratio_sgd_mean",
    "kappa_ratio_muon_mean",
    "n_negative_before_mean",
]].copy()

summary_display = summary_display.rename(columns={
    "n_rows": "n",
    "loss_mean": "loss_mean",
    "lambda_min_pos_before_mean": "lambda_min_pos_before_mean",
    "lambda_min_pos_lift_sgd_mean": "lift_sgd_mean",
    "lambda_min_pos_lift_muon_mean": "lift_muon_mean",
    "muon_to_sgd_ratio_of_means": "muon_to_sgd_ratio_of_means",
    "lambda_min_pos_lift_sgd_median": "lift_sgd_median",
    "lambda_min_pos_lift_muon_median": "lift_muon_median",
    "kappa_ratio_sgd_mean": "kappa_ratio_sgd_mean",
    "kappa_ratio_muon_mean": "kappa_ratio_muon_mean",
    "n_negative_before_mean": "n_negative_before_mean",
})

formatters = {
    "loss_mean": lambda x: f"{x:.4e}",
    "lambda_min_pos_before_mean": lambda x: f"{x:.4e}",
    "lift_sgd_mean": lambda x: f"{x:.4f}",
    "lift_muon_mean": lambda x: f"{x:.4f}",
    "muon_to_sgd_ratio_of_means": lambda x: f"{x:.4f}",
    "lift_sgd_median": lambda x: f"{x:.4f}",
    "lift_muon_median": lambda x: f"{x:.4f}",
    "kappa_ratio_sgd_mean": lambda x: f"{x:.4f}",
    "kappa_ratio_muon_mean": lambda x: f"{x:.4f}",
    "n_negative_before_mean": lambda x: f"{x:.2f}",
}

print(summary_display.to_string(index=False, formatters=formatters))
summary_display


### Interpretation

At this stage, the important caution is that the `Muon/SGD` headline is still a **ratio of stepwise means**, which can become very large when the pre-step positive spectral floor is tiny late in training. That is why the notebook also reports pairwise summaries and the explicit step-100 failure case later on.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))

for seed, group in rows_df.groupby("seed"):
    group = group.sort_values("step")
    ax.plot(
        group["step"] - 4,
        group["lambda_min_pos_lift_sgd"],
        "o",
        color="tab:blue",
        alpha=0.35,
        markersize=5,
    )
    ax.plot(
        group["step"] + 4,
        group["lambda_min_pos_lift_muon"],
        "s",
        color="tab:orange",
        alpha=0.35,
        markersize=5,
    )

ax.plot(
    summary_df["step"],
    summary_df["lambda_min_pos_lift_sgd_mean"],
    "-o",
    color="tab:blue",
    linewidth=2.5,
    label="SGD mean lift",
)
ax.plot(
    summary_df["step"],
    summary_df["lambda_min_pos_lift_muon_mean"],
    "-s",
    color="tab:orange",
    linewidth=2.5,
    label="Muon mean lift",
)
ax.axhline(1.0, color="black", linestyle="--", linewidth=1, label="no lift")
ax.axvline(100, color="tab:red", linestyle=":", linewidth=1.5, label="step 100")
ax.set_yscale("log")
ax.set_xlabel("SGD warmup checkpoint step")
ax.set_ylabel(r"One-step lift in $\lambda_{\min}^+$")
ax.set_title(r"One-step $\lambda_{\min}^+$ lift from matched SGD checkpoints")
ax.set_xticks(summary_df["step"])
ax.legend(loc="best")
plt.tight_layout()
plt.show()


### Interpretation

This figure makes the core result visually transparent:

- Muon often produces much larger one-step increases in $\lambda_{\min}^+$ than SGD.
- The effect is **not uniform**. In particular, the step-100 checkpoint is a clear reversal where Muon underperforms SGD.
- A log scale is useful because the late-checkpoint effects become very large.


In [ ]:
ratio_rows = rows_df[rows_df["muon_to_sgd_lift_ratio"] > 0].copy()
ratio_summary = summary_df[summary_df["muon_to_sgd_ratio_of_means"] > 0].copy()

fig, ax = plt.subplots(figsize=(10, 5.5))

for seed, group in ratio_rows.groupby("seed"):
    group = group.sort_values("step")
    ax.plot(
        group["step"],
        group["muon_to_sgd_lift_ratio"],
        "o",
        color="0.55",
        alpha=0.45,
        markersize=5,
    )

ax.plot(
    ratio_summary["step"],
    ratio_summary["muon_to_sgd_ratio_of_means"],
    "-o",
    color="tab:red",
    linewidth=2.5,
    label="Muon/SGD ratio of step means",
)
ax.axhline(1.0, color="black", linestyle="--", linewidth=1, label="Muon = SGD")
ax.axhline(3.0, color="tab:green", linestyle=":", linewidth=1.5, label="3x reference")
ax.axvline(100, color="tab:red", linestyle=":", linewidth=1.0)
ax.set_yscale("log")
ax.set_xlabel("SGD warmup checkpoint step")
ax.set_ylabel(r"Muon / SGD on one-step $\lambda_{\min}^+$ lift")
ax.set_title(r"Relative one-step lift: Muon versus SGD")
ax.set_xticks(summary_df["step"])
ax.legend(loc="best")
plt.tight_layout()
plt.show()


### Interpretation

This ratio view is the cleanest way to see when Muon beats SGD on the positive spectral floor.

- Values above 1 mean Muon lifted $\lambda_{\min}^+$ more than SGD.
- Values below 1 mean SGD won at that checkpoint.
- The step-100 dip is not a small fluctuation; it is a full paired reversal in the default run.


In [ ]:
step100_df = rows_df.loc[rows_df["step"] == 100, [
    "seed",
    "loss",
    "lambda_min_pos_before",
    "lambda_min_pos_after_sgd",
    "lambda_min_pos_after_muon",
    "lambda_min_pos_lift_sgd",
    "lambda_min_pos_lift_muon",
    "muon_to_sgd_lift_ratio",
    "n_negative_before",
]].copy()

step100_df = step100_df.rename(columns={
    "lambda_min_pos_before": "lambda_min_pos_before",
    "lambda_min_pos_after_sgd": "lambda_min_pos_after_sgd",
    "lambda_min_pos_after_muon": "lambda_min_pos_after_muon",
    "lambda_min_pos_lift_sgd": "lift_sgd",
    "lambda_min_pos_lift_muon": "lift_muon",
    "muon_to_sgd_lift_ratio": "muon_to_sgd_ratio",
})

formatters = {
    "loss": lambda x: f"{x:.4e}",
    "lambda_min_pos_before": lambda x: f"{x:.4e}",
    "lambda_min_pos_after_sgd": lambda x: f"{x:.4e}",
    "lambda_min_pos_after_muon": lambda x: f"{x:.4e}",
    "lift_sgd": lambda x: f"{x:.4f}",
    "lift_muon": lambda x: f"{x:.4f}",
    "muon_to_sgd_ratio": lambda x: f"{x:.4f}",
    "n_negative_before": lambda x: f"{x:.0f}",
}

muon_better_count_step100 = int((step100_df["muon_to_sgd_ratio"] > 1.0).sum())
print(f"Muon beats SGD at step 100 in {muon_better_count_step100}/{len(step100_df)} seeds.")
print(step100_df.to_string(index=False, formatters=formatters))
step100_df


### Interpretation

This is the main local failure mode of the default experiment: at step 100, Muon loses to SGD in every seed. Any serious narrative for this pair therefore has to acknowledge that the effect is strong overall but not monotone or universal.


In [ ]:
spectrum_display = summary_df[[
    "step",
    "lambda_min_raw_before_mean",
    "n_negative_before_mean",
    "n_negative_after_sgd_mean",
    "n_negative_after_muon_mean",
    "n_positive_before_mean",
    "n_near_zero_before_mean",
]].copy()

formatters = {
    "lambda_min_raw_before_mean": lambda x: f"{x:.4e}",
    "n_negative_before_mean": lambda x: f"{x:.2f}",
    "n_negative_after_sgd_mean": lambda x: f"{x:.2f}",
    "n_negative_after_muon_mean": lambda x: f"{x:.2f}",
    "n_positive_before_mean": lambda x: f"{x:.2f}",
    "n_near_zero_before_mean": lambda x: f"{x:.2f}",
}

print(spectrum_display.to_string(index=False, formatters=formatters))

fig, ax = plt.subplots(figsize=(10, 4.8))
ax.plot(summary_df["step"], summary_df["n_negative_before_mean"], "-o", label="before step")
ax.plot(summary_df["step"], summary_df["n_negative_after_sgd_mean"], "--s", label="after 1 SGD step")
ax.plot(summary_df["step"], summary_df["n_negative_after_muon_mean"], ":^", label="after 1 Muon step")
ax.set_xlabel("SGD warmup checkpoint step")
ax.set_ylabel("Mean number of negative Hessian eigenvalues")
ax.set_title("Signed-spectrum diagnostic: Hessian indefiniteness is persistent")
ax.set_xticks(summary_df["step"])
ax.legend(loc="best")
plt.tight_layout()
plt.show()

spectrum_display


### Interpretation

The signed-spectrum diagnostics are the main reason the notebook now uses the notation $\lambda_{\min}^+$ throughout.

Because the Hessian retains multiple negative eigenvalues across the warmup range, the experiment is **not** measuring the algebraic smallest eigenvalue. It is measuring the positive spectral floor within an indefinite spectrum.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))

axes[0].plot(summary_df["step"], summary_df["kappa_ratio_sgd_mean"], "-o", label="SGD")
axes[0].plot(summary_df["step"], summary_df["kappa_ratio_muon_mean"], "-s", label="Muon")
axes[0].axhline(1.0, color="black", linestyle="--", linewidth=1)
axes[0].set_yscale("log")
axes[0].set_xlabel("SGD warmup checkpoint step")
axes[0].set_ylabel(r"$\kappa_{after} / \kappa_{before}$")
axes[0].set_title(r"One-step conditioning change via $\kappa = \lambda_{\max}/\lambda_{\min}^+$")
axes[0].set_xticks(summary_df["step"])
axes[0].legend(loc="best")

axes[1].plot(summary_df["step"], summary_df["lambda_max_ratio_sgd_mean"], "-o", label="SGD")
axes[1].plot(summary_df["step"], summary_df["lambda_max_ratio_muon_mean"], "-s", label="Muon")
axes[1].axhline(1.0, color="black", linestyle="--", linewidth=1)
axes[1].set_xlabel("SGD warmup checkpoint step")
axes[1].set_ylabel(r"$\lambda_{\max,after} / \lambda_{\max,before}$")
axes[1].set_title(r"Companion sharpness view via $\lambda_{\max}$")
axes[1].set_xticks(summary_df["step"])
axes[1].legend(loc="best")

plt.tight_layout()
plt.show()


## Decision rules and conclusion

The original script reported five thresholded checks (T1-T5). They are retained here for continuity, but they should be read as **descriptive decision rules**, not formal statistical hypothesis tests.


In [ ]:
decision_rules = results["decision_rules"]

legacy_checks_df = pd.DataFrame([
    {
        "rule": "T1_majority_steps_muon_better",
        "pass": decision_rules["T1_majority_steps_muon_better"]["pass"],
        "detail": f"{decision_rules['T1_majority_steps_muon_better']['muon_wins_count']}/{decision_rules['T1_majority_steps_muon_better']['total_points']} steps",
    },
    {
        "rule": "T2_mean_step_ratio_gt_3",
        "pass": decision_rules["T2_mean_step_ratio_gt_3"]["pass"],
        "detail": f"mean={decision_rules['T2_mean_step_ratio_gt_3']['mean_ratio']:.4f}, median={decision_rules['T2_mean_step_ratio_gt_3']['median_ratio']:.4f}",
    },
    {
        "rule": "T3_majority_steps_kappa_better",
        "pass": decision_rules["T3_majority_steps_kappa_better"]["pass"],
        "detail": f"{decision_rules['T3_majority_steps_kappa_better']['muon_better_count']}/{decision_rules['T3_majority_steps_kappa_better']['total_points']} steps",
    },
    {
        "rule": "T4_step_ratio_cv_lt_1",
        "pass": decision_rules["T4_step_ratio_cv_lt_1"]["pass"],
        "detail": f"cv={decision_rules['T4_step_ratio_cv_lt_1']['cv']:.4f}",
    },
    {
        "rule": "T5_late_steps_mean_ratio_gt_1",
        "pass": decision_rules["T5_late_steps_mean_ratio_gt_1"]["pass"],
        "detail": f"late mean={decision_rules['T5_late_steps_mean_ratio_gt_1']['mean_late_ratio']:.4f}",
    },
])

print(legacy_checks_df.to_string(index=False))

pairwise = decision_rules["pairwise_ratio_summary"]
step100_check = decision_rules["step_100_reversal_check"]
step100_ratio_of_means = float(summary_df.loc[summary_df["step"] == 100, "muon_to_sgd_ratio_of_means"].iloc[0])
neg_min = float(summary_df["n_negative_before_mean"].min())
neg_max = float(summary_df["n_negative_before_mean"].max())

print("\nConclusion")
print("----------")
print(
    f"- Supported in the default five-seed run: Muon/SGD > 1 at "
    f"{decision_rules['T1_majority_steps_muon_better']['muon_wins_count']}/"
    f"{decision_rules['T1_majority_steps_muon_better']['total_points']} checkpoints, "
    f"with legacy mean ratio {decision_rules['T2_mean_step_ratio_gt_3']['mean_ratio']:.4f}."
)
print(
    f"- However, this is specifically a λ_min^+ result, not a statement about the algebraic minimum eigenvalue."
)
print(
    f"- The Hessian is indefinite throughout the observed warmup range: mean negative-eigenvalue count before the step ranges from {neg_min:.1f} to {neg_max:.1f}."
)
print(
    f"- Step 100 is an explicit reversal: Muon beats SGD in "
    f"{step100_check['muon_better_count']}/{step100_check['total_points']} seeds there, "
    f"and the step-100 ratio of means is {step100_ratio_of_means:.4f}."
)
print(
    f"- Pairwise robustness is weaker than the headline mean suggests: Muon wins "
    f"{pairwise['muon_better_count']}/{pairwise['valid_pairwise_count']} valid seed-step pairs, "
    f"with pairwise median ratio {pairwise['median_ratio']:.4f} and geometric mean {pairwise['geometric_mean_ratio']:.4f}."
)
print(
    "- Not established here: a clean causal decomposition of cumulative conditioning gains, "
    "or a claim about the true algebraic λ_min of the Hessian."
)
print(
    "- Major caveats: finite-difference Hessian, learning-rate mismatch (Muon 0.02 vs SGD 0.01), "
    "shared SGD momentum buffer, only five seeds, and a toy deep linear model."
)

legacy_checks_df
